In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [5]:
article_text="""
Earth is the third planet from the Sun and the only
astronomical object known to harbor life. This is
enabled by Earth being an ocean world, the only one in
the Solar System sustaining liquid surface water. Almost
all of Earth's water is contained in its global ocean,
covering 70.8% of Earth's crust. The remaining 29.2% of
Earth's crust is land, most of which is located in the
form of continental landmasses within Earth's land
hemisphere. Most of Earth's land is at least somewhat
humid and covered by vegetation, while large sheets of
ice at Earth's polar deserts retain more water than
Earth's groundwater, lakes, rivers, and atmospheric
water combined. Earth's crust consists of slowly moving
tectonic plates, which interact to produce mountain
ranges, volcanoes, and earthquakes. Earth has a liquid
outer core that generates a magnetosphere capable of
deflecting most of the destructive solar winds and
cosmic radiation.
Earth has a dynamic atmosphere, which sustains
Earth's surface conditions and protects it from most
meteoroids and UV-light at entry. It has a composition
of primarily nitrogen and oxygen. Water vapor is widely
present in the atmosphere, forming clouds that cover
most of the planet. The water vapor acts as a greenhouse
gas and, together with other greenhouse gases in the
atmosphere, particularly carbon dioxide (CO2), creates
the conditions for both liquid surface water and water
vapor to persist via the capturing of energy from the
Sun's light. This process maintains the current average
surface temperature of 14.76 °C (58.57 °F), at which
water is liquid under normal atmospheric pressure.
Differences in the amount of captured energy between
geographic regions (as with the equatorial region
receiving more sunlight than the polar regions) drive
atmospheric and ocean currents, producing a global
climate system with different climate regions, and a
range of weather phenomena such as precipitation,
allowing components such as nitrogen to cycle.
Earth
The Blue Marble, Apollo 17, December 1972
Designations
Alternative
names
The world · The globe ·
Terra · Tellus · Gaia ·
Mother Earth · Sol III
Adjectives Earthly · Terrestrial · Terran
· Tellurian
Symbol and
Orbital characteristics
Epoch J2000[n 1]
Aphelion 152 097 597 km
Perihelion 147 098 450 km[n 2]
Semi-major axis 149 598 023 km[1]
Eccentricity 0.016 7086[1]
Orbital period
365.256 363 004 d[2]
(sidereal)
(1.000 017 420 96 aj)
29.7827 km/s[3]
Average orbital
speed
Mean anomaly 358.617°
Inclination 7.155°
– Sun's equator;
Earth is rounded into an ellipsoid with a circumference
of about 40,000 kilometres (25,000 miles). It is the
densest planet in the Solar System. Of the four rocky
planets, it is the largest and most massive. Earth is
about eight light-minutes away from the Sun and orbits
it, taking a year (about 365.25 days) to complete one
revolution. Earth rotates around its own axis in slightly
less than a day (in about 23 hours and 56 minutes).
Earth's axis of rotation is tilted with respect to the
perpendicular to its orbital plane around the Sun,
producing seasons. Earth is orbited by one permanent
natural satellite, the Moon, which orbits Earth at
384,400 km (238,900 mi)—1.28 light seconds—and is
roughly a quarter as wide as Earth. The Moon's gravity
helps stabilize Earth's axis, causes tides and gradually
slows Earth's rotation. Tidal locking has made the Moon
always face Earth with the same side.
Earth, like most other bodies in the Solar System,
formed about 4.5 billion years ago from gas and dust in
the early Solar System. During the first billion years of
Earth's history, the ocean formed and then life
developed within it. Life spread globally and has been
altering Earth's atmosphere and surface, leading to the
Great Oxidation Event two billion years ago. Humans
emerged 300,000 years ago in Africa and have spread
across every continent on Earth. Humans depend on
Earth's biosphere and natural resources for their
survival, but have increasingly impacted the planet's
environment. Humanity's current impact on Earth's
climate and biosphere is unsustainable, threatening the
livelihood of humans and many other forms of life, and
causing widespread extinctions.
[23]
"""

In [4]:
# TODO: Read PDF data, feed into Claude
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        },
        "title":"earth.pdf",
        "citations":{"enabled": True }
    },
    {
        "type": "text",
        "text": "How were Earth's atmosphere and oceans were formed?"
    }
])

chat(messages)


Message(id='msg_011Cdqxs2smtYqhGGTL3c8Pm', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', ro